# Очистка данных и Feature Engineering

Идея и что я хочу тут реализовать

1. Загрузить и оптимизировать по памяти 12.8 млн строк дампа IMDb.
2. Очистить данные от невалидных значений.
3. Сгенерировать признаки популярности и аномальности (`log_votes`, `rating_to_log_votes`).
4. Сформировать таргет редкого класса (0.17%) и сохранить очищенный датасет.

In [11]:
#Основные импорты
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Проверяем

print(f"Pandas: {pd.__version__}, NumPy: {np.__version__}")

Pandas: 3.0.6, NumPy: 2.5.4


### 1. Загрузка метаданных (title.basics) и оптимизация памяти
В сыром дампе 12.8 млн записей. Оптимизируем потребление RAM:
- Читаем только необходимые столбцы (`usecols`)
- Указываем строгие типы данных (`dtype`), переводим `titleType` в `category`, если не указывать просто все будут обджект, а это самый плохой тип
- Сразу фильтруем только полнометражные фильмы (`titleType == 'movie'`) и освобождаем память.

In [12]:
basics = pd.read_csv(
    'data/title.basics.tsv.gz',
    sep='\t',
    na_values=r'\N',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres'],
    dtype={
        'tconst': 'str',
        'titleType': 'category',
        'primaryTitle': 'str',
        'startYear': 'str',
        'runtimeMinutes': 'str',
        'genres': 'str'
    }
)

# Только фильмы
movies = basics[basics['titleType'] == 'movie'].copy()

# Память
del basics

print(f"Полнометражных фильмов отобрано: {len(movies):,}")
movies.head()

Полнометражных фильмов отобрано: 759,363


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,1894,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905,100,NaN
570,tt0000574,movie,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,1907,90,Drama


### 2. Загрузка рейтингов (title.ratings) и объединение (Inner Join)
В дампе рейтингов 1.7 млн записей. Мы выполняем объединение по первичному ключу `tconst`:
- Используем типы пониженной разрядности: `averageRating` (`float32`) и `numVotes` (`int32`)
- Применяем `how='inner'`: отсекаем фильмы без оценок, так как без голосов невозможно выявлять паттерны и аномалии
- Удаляем вспомогательную таблицу `ratings` после объединения.

In [13]:
ratings = pd.read_csv(
    'data/title.ratings.tsv.gz',
    sep='\t',
    na_values=r'\N',
    dtype={'tconst': 'str', 'averageRating': 'float32', 'numVotes': 'int32'}
)

print(f"Всего тайтлов с оценками: {len(ratings):,}")

# JOIN
df = movies.merge(ratings, on='tconst', how='inner')

# Очистка
del ratings
del movies

print(f"Полнометражных фильмов с оценками: {len(df):,}")
df.info()
df.head()

Всего тайтлов с оценками: 1,720,209
Полнометражных фильмов с оценками: 350,001
<class 'pandas.DataFrame'>
RangeIndex: 350001 entries, 0 to 350000
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype   
---  ------          --------------   -----   
 0   tconst          350001 non-null  str     
 1   titleType       350001 non-null  category
 2   primaryTitle    349999 non-null  str     
 3   startYear       349967 non-null  str     
 4   runtimeMinutes  320147 non-null  str     
 5   genres          338530 non-null  str     
 6   averageRating   350001 non-null  float32 
 7   numVotes        350001 non-null  int32   
dtypes: category(1), float32(1), int32(1), str(5)
memory usage: 16.4 MB


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000009,movie,Miss Jerry,1894,45,Romance,5.3,239
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport",5.3,614
2,tt0000502,movie,Bohemios,1905,100,NaN,3.7,27
3,tt0000574,movie,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography",6.0,1100
4,tt0000591,movie,The Prodigal Son,1907,90,Drama,5.0,42


### 3. Очистка данных и приведение типов
В объединенном датасете есть фильмы с невалидными или пропущенными значениями:
- Переводим `startYear` и `runtimeMinutes` в числовой формат через `pd.to_numeric(..., errors='coerce')`
- Фильтруем валидный исторический диапазон годов (1890 – 2026) и хронометраж (20 – 400 минут)
- Исключаем фильмы с отсутствующим жанром
- Приводим типы к `int16` для минимизации расхода памяти (13.4 МБ на 313 тыс. строк).

In [14]:
# Безопасное приведение к числовым типам (ошибки парсинга становятся NaN)
df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce')
df['runtimeMinutes'] = pd.to_numeric(df['runtimeMinutes'], errors='coerce')

# Подсчет пропусков перед фильтрацией
print("Пропуски до фильтрации:")
print(df[['startYear', 'runtimeMinutes', 'genres']].isna().sum())

# Фильтрация валидных диапазонов
mask = (
    df['startYear'].between(1890, 2026) &
    df['runtimeMinutes'].between(20, 400) &
    df['genres'].notna()
)
df = df[mask].copy()

# Оптимизация памяти (int16 занимает 2 байта на число)
df['startYear'] = df['startYear'].astype('int16')
df['runtimeMinutes'] = df['runtimeMinutes'].astype('int16')

print(f"\nФильмов после очистки: {len(df):,}")
df.info()
df.head()

Пропуски до фильтрации:
startYear            34
runtimeMinutes    29854
genres            11471
dtype: int64

Фильмов после очистки: 313,025
<class 'pandas.DataFrame'>
Index: 313025 entries, 0 to 350000
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype   
---  ------          --------------   -----   
 0   tconst          313025 non-null  str     
 1   titleType       313025 non-null  category
 2   primaryTitle    313024 non-null  str     
 3   startYear       313025 non-null  int16   
 4   runtimeMinutes  313025 non-null  int16   
 5   genres          313025 non-null  str     
 6   averageRating   313025 non-null  float32 
 7   numVotes        313025 non-null  int32   
dtypes: category(1), float32(1), int16(2), int32(1), str(3)
memory usage: 13.4 MB


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000009,movie,Miss Jerry,1894,45,Romance,5.3,239
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport",5.3,614
3,tt0000574,movie,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography",6.0,1100
4,tt0000591,movie,The Prodigal Son,1907,90,Drama,5.0,42
8,tt0000679,movie,The Fairylogue and Radio-Plays,1908,120,"Adventure,Fantasy",5.0,85


### 4. Feature Engineering: признаки популярности и аномальности оценок
Создаём ключевые признаки для поиска аномалий и обучения моделей:
- `log_votes`: логарифмирование $\log_{10}(\text{numVotes})$ сжимает гигантский разброс голосов (от 5 до 3 млн) в компактную шкалу ($0.7 \dots 6.5$)
- `rating_to_log_votes`: отношение рейтинга к логарифму голосов — индикатор аномальности (высокий балл при единичных оценках)
- `primary_genre`: основной жанр фильма в типе `category`
- `movie_age`: возраст фильма в годах.

In [16]:
# 1. Логарифм голосов для сжатия тяжелого хвоста распределения
df['log_votes'] = np.log10(df['numVotes']).astype('float32')

# 2. Индикатор аномальности: отношение рейтинга к логарифму голосов (выявление накруток)
df['rating_to_log_votes'] = (df['averageRating'] / (df['log_votes'] + 1)).astype('float32')

# 3. Извлечение первого (основного) жанра и приведение к category
df['primary_genre'] = df['genres'].str.split(',').str[0].astype('category')

# 4. Возраст фильма в годах
df['movie_age'] = (2026 - df['startYear']).astype('int16')

print(f"Новая размерность датасета: {df.shape}")
df[['primaryTitle', 'averageRating', 'numVotes', 'log_votes', 'rating_to_log_votes', 'primary_genre', 'movie_age']].head()

Новая размерность датасета: (313025, 12)


,primaryTitle,averageRating,numVotes,log_votes,rating_to_log_votes,primary_genre,movie_age
0,Miss Jerry,5.3,239,2.378398,1.568791,Romance,132
1,The Corbett-Fitzsimmons Fight,5.3,614,2.788168,1.399093,Documentary,129
3,The Story of the Kelly Gang,6.0,1100,3.041393,1.484637,Action,120
4,The Prodigal Son,5.0,42,1.623249,1.906033,Drama,119
8,The Fairylogue and Radio-Plays,5.0,85,1.929419,1.706823,Adventure,118
